In [32]:
import pandas as pd
from pathlib import Path

In [34]:
DATA_DIR = Path("../data/raw/mine/extended_streaming_history")

COLUMN_MAP = {          # basic-export names -> my canonical names
    "master_metadata_album_artist_name": "artist",
    "master_metadata_track_name": "track",
    "master_metadata_album_album_name": "album",
}

TIMEZONE = "America/Chicago"

files = sorted(DATA_DIR.glob("Streaming_History_Audio*.json"))
files

[PosixPath('../data/raw/mine/extended_streaming_history/Streaming_History_Audio_2021.json'),
 PosixPath('../data/raw/mine/extended_streaming_history/Streaming_History_Audio_2022.json'),
 PosixPath('../data/raw/mine/extended_streaming_history/Streaming_History_Audio_2023.json'),
 PosixPath('../data/raw/mine/extended_streaming_history/Streaming_History_Audio_2024.json'),
 PosixPath('../data/raw/mine/extended_streaming_history/Streaming_History_Audio_2025.json'),
 PosixPath('../data/raw/mine/extended_streaming_history/Streaming_History_Audio_2026.json')]

In [42]:
df = pd.concat([pd.read_json(f) for f in files], ignore_index=True)
df = df.rename(columns=COLUMN_MAP)

print("raw:", len(df))

df = df.drop(columns=["ip_addr"])                 # privacy: gone before anything else touches it

podcasts = df[df["track"].isna()].copy()          # podcasts + audiobooks have no track name
df = df[df["track"].notna()]
print("music only:", len(df), f"({len(podcasts)} podcast/audiobook rows set aside)")

df = df[df["ms_played"] > 0]
print("after zero-ms drop:", len(df))



raw: 61230
music only: 57922 (3308 podcast/audiobook rows set aside)
after zero-ms drop: 57203


In [36]:
print(df.shape)
print(df.head())
print(df.dtypes)

(61230, 23)
                     ts                 platform  ms_played conn_country  \
0  2021-12-02T20:39:08Z  iOS 15.1.1 (iPhone14,2)     129941           US   
1  2021-12-02T20:39:10Z  iOS 15.1.1 (iPhone14,2)       1950           US   
2  2021-12-02T20:39:16Z  iOS 15.1.1 (iPhone14,2)       5340           US   
3  2021-12-02T20:39:39Z  iOS 15.1.1 (iPhone14,2)      22685           US   
4  2021-12-02T20:42:04Z  iOS 15.1.1 (iPhone14,2)     145244           US   

          ip_addr                                   track         artist  \
0  174.253.66.250              Killshot - Slowed + Reverb  Magdalena Bay   
1  174.253.66.250                                Killshot  Magdalena Bay   
2  174.253.66.250                     Secrets (Your Fire)  Magdalena Bay   
3  174.253.66.250                         Mercurial World  Magdalena Bay   
4  174.253.66.250  BIG PERSONA (feat. Tyler, The Creator)     Maxo Kream   

                                    album  \
0                            

In [38]:
df["ts"] = pd.to_datetime(df["ts"])
df['ts'] = pd.to_datetime(df['ts'], utc=True).dt.tz_convert(TIMEZONE)
df["ts"].min(), df["ts"].max()

(Timestamp('2021-12-02 14:39:08-0600', tz='America/Chicago'),
 Timestamp('2026-07-31 18:19:38-0500', tz='America/Chicago'))

In [30]:
print(df.isna().sum())
print(df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values("ts").head(10) #no dupes

ts           0
artist       0
track        0
ms_played    0
dtype: int64
0


,ts,artist,track,ms_played


In [40]:
print((df['ms_played']/60000).describe())
print((df['ms_played'] < 30_000).mean())  # 30k ms = 30 sec

count    61230.000000
mean         2.458133
std          5.739070
min          0.000000
25%          0.042567
50%          0.961600
75%          2.972804
max         99.125417
Name: ms_played, dtype: float64
0.45709619467581253
